# Build the mmcv wheel for current Colab (maintainer, run once)

Colab moved to a Python/torch that OpenMMLab publishes no `mmcv` wheels
for, which broke the training notebooks. This notebook does ONE thing:
compile `mmcv 2.2.0` against the **stock** Colab runtime (nothing
downgraded), verify it actually works, and hand you the `.whl`.

We then commit that wheel to `colab_wheels/` on the `pro` branch of
`dorna-robotics/dorna_vision`, and every training notebook installs it
in seconds from the raw URL.

**Use a GPU runtime** (same as customers train on) so the CUDA ops are
compiled. Expect the build to take ~30-60 min.

In [ ]:
import sys, glob, os, json, urllib.request
print("python:", sys.version)
import torch, numpy
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| numpy:", numpy.__version__)

# modern build tooling — old setuptools breaks on Python 3.13
!pip -q install "setuptools>=79,<80" "wheel>=0.45" ninja
!pip -q install mmengine    # plain pip; NEVER mim (drags openxlab -> setuptools downgrade)

# Fetch the mmcv source DIRECTLY from PyPI — pip runs the (crashing)
# metadata step even for `pip download`, so pip never touches the sdist.
!rm -rf /content/src && mkdir -p /content/src /content/mmcv_wheel
meta = json.load(urllib.request.urlopen("https://pypi.org/pypi/mmcv/2.2.0/json"))
sdist = [u["url"] for u in meta["urls"] if u["filename"].endswith(".tar.gz")][0]
urllib.request.urlretrieve(sdist, "/content/src/mmcv-2.2.0.tar.gz")
!cd /content/src && tar xf mmcv-2.2.0.tar.gz
SRC = "/content/src/mmcv-2.2.0"
assert os.path.isdir(SRC), "sdist fetch failed"

# PY3.13 FIX (PEP 667): setup.py's get_version() execs mmcv/version.py and
# reads locals()['__version__'] — function locals are a snapshot now, so it
# KeyErrors. Exec into an explicit dict instead.
sp = os.path.join(SRC, "setup.py")
src_txt = open(sp).read()
broken = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
fixed = ("        _ns = {}\n"
         "        exec(compile(f.read(), version_file, 'exec'), _ns)\n"
         "    return _ns['__version__']")
assert broken in src_txt, "setup.py shape changed — patch target not found"
open(sp, "w").write(src_txt.replace(broken, fixed))
print("setup.py get_version patched for Python 3.13")

# THE step that has been failing, run bare so its REAL traceback prints:
print("--- egg_info dry run (the real error, if any, is right below) ---")
!cd {SRC} && python setup.py egg_info 2>&1 | tail -40
print("--- end dry run ---")

# Build the wheel. Arch list covers the Colab/customer GPUs (T4 7.5,
# A100 8.0, L4 8.9, +PTX for newer) so ONE hosted wheel serves all.
os.environ["MMCV_WITH_OPS"] = "1"
os.environ["FORCE_CUDA"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5 8.0 8.6 8.9 9.0+PTX"
os.environ["MAX_JOBS"] = str(os.cpu_count())
!cd {SRC} && pip wheel . --no-deps --no-build-isolation -w /content/mmcv_wheel

built = glob.glob("/content/mmcv_wheel/mmcv-2.2.0*.whl")
assert built, "BUILD FAILED — the dry-run block above holds the actual error; paste it back"
print("BUILT:", built[0])

In [ ]:
# Install what we just built and prove the compiled ops load
import glob
built = glob.glob("/content/mmcv_wheel/mmcv-2.2.0*.whl")[0]
!pip -q install {built}
!pip -q install mmdet==3.3.0

# mmdet caps mmcv at <2.2.0 — lift the cap in its version check
import importlib.util, os
spec = importlib.util.find_spec("mmdet")
init_path = os.path.join(os.path.dirname(spec.origin), "__init__.py")
!sed -i "s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/" {init_path}

import importlib
importlib.invalidate_caches()
import torch, mmcv, mmengine, mmdet
from mmcv.ops import MultiScaleDeformableAttention   # the compiled CUDA/C++ part
x = torch.rand(1, 3, 32, 32)
print("mmcv:", mmcv.__version__, "| mmengine:", mmengine.__version__, "| mmdet:", mmdet.__version__)
print("compiled ops import: OK")
if torch.cuda.is_available():
    from mmcv.ops import nms
    import torch as t
    boxes = t.tensor([[0,0,10,10],[1,1,11,11]], dtype=t.float32, device="cuda")
    scores = t.tensor([0.9, 0.8], device="cuda")
    print("CUDA nms op runs:", nms(boxes, scores, 0.5)[0].shape)
print("\nALL GOOD — download the wheel below and commit it to colab_wheels/ (pro branch)")

In [ ]:
from google.colab import files
import glob
files.download(glob.glob("/content/mmcv_wheel/mmcv-2.2.0*.whl")[0])